In [2]:
import pandas as pd
import datetime as dt
import requests
import pandas_datareader.data as web
from lxml import etree

In [3]:
BASE_URL = 'https://sdmx.oecd.org/public/rest/data'
NS = {
    "message": "http://www.sdmx.org/resources/sdmxml/schemas/v2_1/message",
    "generic": "http://www.sdmx.org/resources/sdmxml/schemas/v2_1/data/generic",
    "common": "http://www.sdmx.org/resources/sdmxml/schemas/v2_1/common",
}

In [4]:
def request_oecd_xml(flow_ref, key, params=None, timeout=30):
    if params is None:
        params = {}

    url = f"https://sdmx.oecd.org/public/rest/data/{flow_ref}/{key}"
    headers = {
        # XMLを明示要求（未指定だと形式が変わる場合あり）
        "Accept": "application/vnd.sdmx.genericdata+xml; charset=utf-8; version=2.1"
    }

    resp = requests.get(url, params=params, headers=headers, timeout=timeout)
    resp.raise_for_status()
    return resp

In [5]:
def sdmx_ml_to_dataframe(xml_bytes):
    root = etree.fromstring(xml_bytes)
    rows = []

    # Series単位で走査
    for series in root.xpath(".//generic:Series", namespaces=NS):
        series_key = {}

        # SeriesKeyの各次元
        for v in series.xpath("./generic:SeriesKey/generic:Value", namespaces=NS):
            series_key[v.get("id")] = v.get("value")

        # Obs単位で展開
        for obs in series.xpath("./generic:Obs", namespaces=NS):
            row = dict(series_key)

            # 時間軸（ObsDimension）
            obs_dim = obs.xpath("./generic:ObsDimension", namespaces=NS)
            if obs_dim:
                row["TIME_PERIOD"] = obs_dim[0].get("value")

            # 値（ObsValue）
            obs_val = obs.xpath("./generic:ObsValue", namespaces=NS)
            if obs_val:
                row["OBS_VALUE"] = obs_val[0].get("value")

            # 観測属性（あれば）
            for a in obs.xpath("./generic:Attributes/generic:Value", namespaces=NS):
                row[a.get("id")] = a.get("value")

            rows.append(row)

    df = pd.DataFrame(rows)

    # 数値変換（変換できない値はNaN）
    if "OBS_VALUE" in df.columns:
        df["OBS_VALUE"] = pd.to_numeric(df["OBS_VALUE"], errors="coerce")

    return df

In [6]:

"""
生産性と単位労働コスト
OECD生産性データベースは、ユーザーに最も包括的で最新の生産性推計値を提供することを目的としています。
更新サイクルはローリング方式で、データセット内の各変数は、ソースデータベースで更新されるとすぐに公開されます。
生産性データベースには、雇用または労働時間で測定された労働生産性と、資本および労働投入の構成要素に関するデータが含まれています。
生産性データベースは、水準別、成長率別、産業別に年間データを提供していますが、生産性および単位労働コストに関するデータベースは四半期推計値です。
すべてのデータセットおよび方法論に関する詳細情報は、添付ファイルOECD-Productivity-Statistics-Database-metadataに記載されています。
https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_PDB@DF_PDB_ULC_Q,1.0/.Q.......?startPeriod=2024-Q4
"""
response = request_oecd_xml(
    flow_ref="OECD.SDD.TPS,DSD_PDB@DF_PDB_ULC_Q,1.0",
    key=".Q.......",
    params={"startPeriod": "1990-Q1"},
)
df = sdmx_ml_to_dataframe(response.content)
df.head()

,REF_AREA,FREQ,MEASURE,ACTIVITY,UNIT_MEASURE,PRICE_BASE,TRANSFORMATION,ADJUSTMENT,CONVERSION_TYPE,TIME_PERIOD,OBS_VALUE,OBS_STATUS
0,USA,Q,ULCE,_T,PP,V,G1,S,NC,1990-Q1,0.693625,A
1,USA,Q,ULCE,_T,PP,V,G1,S,NC,1990-Q2,1.662272,A
2,USA,Q,ULCE,_T,PP,V,G1,S,NC,1990-Q3,1.054212,A
3,USA,Q,ULCE,_T,PP,V,G1,S,NC,1990-Q4,1.323731,A
4,USA,Q,ULCE,_T,PP,V,G1,S,NC,1991-Q1,0.672435,A


In [7]:
# このレスポンスで使われている主なコードを日本語ラベルへ展開
code_labels_ja = {
    "REF_AREA": {
        "AUS": "オーストラリア", "AUT": "オーストリア", "BEL": "ベルギー", "CAN": "カナダ",
        "CHE": "スイス", "CRI": "コスタリカ", "CZE": "チェコ", "DEU": "ドイツ", "DNK": "デンマーク",
        "EA19": "ユーロ圏（19か国）", "ESP": "スペイン", "EST": "エストニア", "FIN": "フィンランド",
        "FRA": "フランス", "GBR": "英国", "GRC": "ギリシャ", "HUN": "ハンガリー", "IRL": "アイルランド",
        "ISR": "イスラエル", "ITA": "イタリア", "JPN": "日本", "KOR": "韓国", "LTU": "リトアニア",
        "LUX": "ルクセンブルク", "LVA": "ラトビア", "NLD": "オランダ", "NOR": "ノルウェー", "NZL": "ニュージーランド",
        "OECD": "OECD合計", "POL": "ポーランド", "PRT": "ポルトガル", "SVK": "スロバキア", "SVN": "スロベニア",
        "SWE": "スウェーデン", "TUR": "トルコ", "USA": "米国", "ZAF": "南アフリカ"
    },
    "FREQ": {"Q": "四半期"},
    "MEASURE": {
        "ULCE": "単位労働コスト（雇用者ベース）",
        "LCEMP": "就業者1人あたり労働報酬",
        "GDPEMP": "就業者1人あたりGDP（労働生産性）",
    },
    "ACTIVITY": {"_T": "全産業（総計）"},
    "UNIT_MEASURE": {
        "IX": "指数",
        "PA": "前年同期比（%）",
        "PP": "前期比（%）",
    },
    "PRICE_BASE": {
        "V": "名目（Current prices）",
        "Q": "実質（Volume / Constant prices）",
    },
    "TRANSFORMATION": {
        "_Z": "水準（指数）",
        "G1": "前期比",
        "GY": "前年同期比",
    },
    "ADJUSTMENT": {
        "S": "季節調整済み",
        "N": "季節調整なし",
    },
    "CONVERSION_TYPE": {"NC": "標準変換"},
    "OBS_STATUS": {
        "A": "通常値（有効）",
    },
}


def add_labels(df_in, label_map):
    out = df_in.copy()
    for col, cmap in label_map.items():
        if col in out.columns:
            out[f"{col}_JA"] = out[col].map(cmap).fillna(out[col])
    return out


def infer_obs_value_unit_ja(unit_measure, transformation):
    if unit_measure == "IX":
        return "指数（基準年=100の指数系列）"
    if unit_measure == "PA":
        return "前年同期比（%）"
    if unit_measure == "PP":
        if transformation == "G1":
            return "前期比（%）"
        return "変化率（%）"
    return unit_measure


# 日本語ラベル展開
df_labeled = add_labels(df, code_labels_ja)
df_labeled["OBS_VALUE_UNIT_JA"] = [
    infer_obs_value_unit_ja(u, t)
    for u, t in zip(df_labeled["UNIT_MEASURE"], df_labeled["TRANSFORMATION"])
]

# コード表（このレスポンスで実際に出現したコードのみ）
codebook_rows = []
for dim, cmap in code_labels_ja.items():
    if dim in df_labeled.columns:
        for code in sorted(df_labeled[dim].dropna().unique()):
            codebook_rows.append({
                "dimension": dim,
                "code": code,
                "label_ja": cmap.get(code, code),
            })

codebook_df = pd.DataFrame(codebook_rows)
print("=== コード表（日本語ラベル） ===")
display(codebook_df.sort_values(["dimension", "code"]).reset_index(drop=True))

print("=== ラベル付きデータ（先頭） ===")
display(
    df_labeled[
        [
            "REF_AREA", "REF_AREA_JA", "MEASURE", "MEASURE_JA", "ADJUSTMENT", "ADJUSTMENT_JA",
            "UNIT_MEASURE", "TRANSFORMATION", "TIME_PERIOD", "OBS_VALUE", "OBS_VALUE_UNIT_JA", "OBS_STATUS"
        ]
    ].head(20)
)

print("=== OBS_VALUEの単位（このレスポンス内） ===")
display(df_labeled["OBS_VALUE_UNIT_JA"].value_counts().rename_axis("unit").reset_index(name="count"))

=== コード表（日本語ラベル） ===


,dimension,code,label_ja
0,ACTIVITY,_T,全産業（総計）
1,ADJUSTMENT,N,季節調整なし
2,ADJUSTMENT,S,季節調整済み
3,CONVERSION_TYPE,NC,標準変換
4,FREQ,Q,四半期
5,MEASURE,GDPEMP,就業者1人あたりGDP（労働生産性）
6,MEASURE,LCEMP,就業者1人あたり労働報酬
7,MEASURE,ULCE,単位労働コスト（雇用者ベース）
8,OBS_STATUS,A,通常値（有効）
9,PRICE_BASE,Q,実質（Volume / Constant prices）


=== ラベル付きデータ（先頭） ===


,REF_AREA,REF_AREA_JA,MEASURE,MEASURE_JA,ADJUSTMENT,ADJUSTMENT_JA,UNIT_MEASURE,TRANSFORMATION,TIME_PERIOD,OBS_VALUE,OBS_VALUE_UNIT_JA,OBS_STATUS
0,USA,米国,ULCE,単位労働コスト（雇用者ベース）,S,季節調整済み,PP,G1,1990-Q1,0.693625,前期比（%）,A
1,USA,米国,ULCE,単位労働コスト（雇用者ベース）,S,季節調整済み,PP,G1,1990-Q2,1.662272,前期比（%）,A
2,USA,米国,ULCE,単位労働コスト（雇用者ベース）,S,季節調整済み,PP,G1,1990-Q3,1.054212,前期比（%）,A
3,USA,米国,ULCE,単位労働コスト（雇用者ベース）,S,季節調整済み,PP,G1,1990-Q4,1.323731,前期比（%）,A
4,USA,米国,ULCE,単位労働コスト（雇用者ベース）,S,季節調整済み,PP,G1,1991-Q1,0.672435,前期比（%）,A
5,USA,米国,ULCE,単位労働コスト（雇用者ベース）,S,季節調整済み,PP,G1,1991-Q2,0.554133,前期比（%）,A
6,USA,米国,ULCE,単位労働コスト（雇用者ベース）,S,季節調整済み,PP,G1,1991-Q3,0.692192,前期比（%）,A
7,USA,米国,ULCE,単位労働コスト（雇用者ベース）,S,季節調整済み,PP,G1,1991-Q4,1.014027,前期比（%）,A
8,USA,米国,ULCE,単位労働コスト（雇用者ベース）,S,季節調整済み,PP,G1,1992-Q1,0.651012,前期比（%）,A
9,USA,米国,ULCE,単位労働コスト（雇用者ベース）,S,季節調整済み,PP,G1,1992-Q2,0.401603,前期比（%）,A


=== OBS_VALUEの単位（このレスポンス内） ===


,unit,count
0,指数（基準年=100の指数系列）,26666
1,前期比（%）,13804
2,前年同期比（%）,13556


In [10]:

# -------------------------------------------------------------------
# PDB_LV: 就業者1人あたりGDP（実質・USD PPP換算）年次データ
# 新SDMX API（DF_PDB_LV）を使用してG7の1990〜2024年を取得
# MEASURE=GDPEMP, UNIT_MEASURE=USD_PPP_PS, PRICE_BASE=Q
# 同一(国, 年)が複数ビンテージで存在する場合は最新値を採用
# -------------------------------------------------------------------
G7_CODES = ["CAN", "FRA", "DEU", "ITA", "JPN", "GBR", "USA"]

resp_lv = request_oecd_xml(
    flow_ref="OECD.SDD.TPS,DSD_PDB@DF_PDB_LV,1.0",
    key="+".join(G7_CODES) + ".A.GDPEMP._T.USD_PPP_PS......",
    params={"startPeriod": "1990", "endPeriod": "2024"},
)
df_lv_raw = sdmx_ml_to_dataframe(resp_lv.content)

# 実質（Constant prices = Q）に絞り、(国, 年)の重複は最新ビンテージを採用
df_lv = (
    df_lv_raw[df_lv_raw["PRICE_BASE"] == "Q"]
    .sort_values(["REF_AREA", "TIME_PERIOD"])
    .drop_duplicates(subset=["REF_AREA", "TIME_PERIOD"], keep="last")
    .assign(TIME_PERIOD=lambda d: pd.to_numeric(d["TIME_PERIOD"], errors="coerce").astype("Int64"))
    .sort_values(["REF_AREA", "TIME_PERIOD"])
    .reset_index(drop=True)
)

print(f"取得行数: {len(df_lv):,}  |  国/地域: {df_lv['REF_AREA'].nunique()}  |  期間: {df_lv['TIME_PERIOD'].min()}〜{df_lv['TIME_PERIOD'].max()}")
print("\n各国の年数:")
display(
    df_lv.groupby("REF_AREA")["TIME_PERIOD"]
    .agg(年数="count", 開始年="min", 終了年="max")
    .reset_index()
)


取得行数: 242  |  国/地域: 7  |  期間: 1990〜2024

各国の年数:


,REF_AREA,年数,開始年,終了年
0,CAN,35,1990,2024
1,DEU,34,1991,2024
2,FRA,35,1990,2024
3,GBR,35,1990,2024
4,ITA,35,1990,2024
5,JPN,34,1990,2023
6,USA,34,1990,2023


In [11]:
import plotly.express as px

# G7加盟国
G7 = {
    "CAN": "カナダ",
    "FRA": "フランス",
    "DEU": "ドイツ",
    "ITA": "イタリア",
    "JPN": "日本",
    "GBR": "英国",
    "USA": "米国",
}

df_g7 = (
    df_lv[df_lv["REF_AREA"].isin(G7)]
    .sort_values(["TIME_PERIOD", "REF_AREA"])
    .assign(国=lambda d: d["REF_AREA"].map(G7))
    .reset_index(drop=True)
)

fig = px.line(
    df_g7,
    x="TIME_PERIOD",
    y="OBS_VALUE",
    color="国",
    markers=True,
    title="G7 就業者1人あたりGDP（実質・USD PPP換算）",
    labels={
        "TIME_PERIOD": "年",
        "OBS_VALUE": "USD（PPP換算、実質）",
        "国": "国",
    },
    color_discrete_sequence=px.colors.qualitative.G10,
)

fig.update_traces(marker=dict(size=4), line=dict(width=2))
fig.update_layout(
    xaxis=dict(dtick=5, tickangle=0),
    yaxis=dict(tickformat=",.0f"),
    legend=dict(title="国"),
    hovermode="x unified",
    width=900,
    height=520,
)
fig.show()
